In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

CUDA available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM (GB): 8.546484224


In [1]:
import pandas as pd
import numpy as np
import json
import random
import os
import sys

In [8]:
trans = pd.read_csv('./ieee-fraud-detection/train_transaction.csv')
identity = pd.read_csv('./ieee-fraud-detection/train_identity.csv')
print(trans.shape, identity.shape)

(590540, 394) (144233, 41)


In [9]:
df = trans.merge(identity, how='left', on='TransactionID')
print(df.shape)


(590540, 434)


In [10]:
df.head()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [11]:
print(df['isFraud'].value_counts(normalize=True))

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64


In [12]:
missing_pct = df.isnull().mean().sort_values(ascending = False)
print(missing_pct.head(20))

id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
D7       0.934099
id_18    0.923607
D13      0.895093
D14      0.894695
D12      0.890410
id_04    0.887689
id_03    0.887689
D6       0.876068
id_33    0.875895
id_09    0.873123
dtype: float64


In [13]:
# for human readable columns, we can use the following list of columns that are not anonymized or engineered. The rest of the columns are anonymized or engineered features.
nameable = ['TransactionAmt', 'ProductCD', 'card4', 'card6', 
            'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo',
            'addr1', 'addr2', 'dist1', 'dist2']
nameable = [c for c in nameable if c in df.columns]

anonymized = [c for c in df.columns if c.startswith('V') or c.startswith('id_') or c.startswith('C') or c.startswith('D') or c.startswith('M')]

print(f"Nameable (human-readable): {len(nameable)}")
print(f"Anonymized/engineered: {len(anonymized)}")
print(f"Total columns: {df.shape[1]}")

Nameable (human-readable): 12
Anonymized/engineered: 417
Total columns: 434


In [14]:
df.to_pickle('./ieee-fraud-detection/merged_raw.pkl')
print("Saved.")

Saved.


In [4]:
# another day rerunning code 23/7/26 this block is for rerunning dont get confused
df = pd.read_pickle('./ieee-fraud-detection/merged_raw.pkl')
print(df.shape)
df.head()

(590540, 434)


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


In [5]:
# --- IGNORE ---
df['isFraud'].value_counts(normalize = True)

isFraud
0    0.96501
1    0.03499
Name: proportion, dtype: float64

In [6]:
# --- IGNORE ---
missing_pct = df.isnull().mean().sort_values(ascending = False)
print(missing_pct.head(20))

id_24    0.991962
id_25    0.991310
id_07    0.991271
id_08    0.991271
id_21    0.991264
id_26    0.991257
id_27    0.991247
id_23    0.991247
id_22    0.991247
dist2    0.936284
D7       0.934099
id_18    0.923607
D13      0.895093
D14      0.894695
D12      0.890410
id_04    0.887689
id_03    0.887689
D6       0.876068
id_33    0.875895
id_09    0.873123
dtype: float64


In [7]:
# --- IGNORE ---
# for human readable columns, we can use the following list of columns that are not anonymized or engineered. The rest of the columns are anonymized or engineered features.
nameable = ['TransactionAmt', 'ProductCD', 'card4', 'card6', 
            'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo',
            'addr1', 'addr2', 'dist1', 'dist2']
nameable = [c for c in nameable if c in df.columns]

anonymized = [c for c in df.columns if c.startswith('V') or c.startswith('id_') or c.startswith('C') or c.startswith('D') or c.startswith('M')]

print(f"Nameable (human-readable): {len(nameable)}")
print(f"Anonymized/engineered: {len(anonymized)}")
print(f"Total columns: {df.shape[1]}")

Nameable (human-readable): 12
Anonymized/engineered: 417
Total columns: 434


In [8]:
df = df.sort_values(by = 'TransactionDT')
split_index = int(df.shape[0] * 0.8)
train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (472432, 434)
Test shape: (118108, 434)


In [8]:
categorical_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo', 'addr1', 'addr2']
categorical_cols = [c for c in categorical_cols if c in df.columns]

for c in categorical_cols:
    df[c] = df[c].astype('category') 

In [9]:
drop_cols = ['TransactionID', 'TransactionDT', 'isFraud']
features = [c for c in df.columns if c not in drop_cols]
# X_train = train_df[features]
# y_train = train_df['isFraud']
# X_test = test_df[features]
# y_test = test_df['isFraud']

In [22]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print("X_train shape:", X_train.shape)

print("X_test shape:", X_test.shape)

X_train shape: (472432, 431)
X_test shape: (118108, 431)


In [23]:
X_train.to_pickle('./ieee-fraud-detection/X_train.pkl')
X_test.to_pickle('./ieee-fraud-detection/X_test.pkl')
y_train.to_pickle('./ieee-fraud-detection/y_train.pkl')
y_test.to_pickle('./ieee-fraud-detection/y_test.pkl')

In [10]:
X_train = pd.read_pickle('./ieee-fraud-detection/X_train.pkl')
X_test = pd.read_pickle('./ieee-fraud-detection/X_test.pkl')
y_train = pd.read_pickle('./ieee-fraud-detection/y_train.pkl')
y_test = pd.read_pickle('./ieee-fraud-detection/y_test.pkl')
print("Reloaded Data shapes: ", X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Reloaded Data shapes:  (472432, 431) (118108, 431) (472432,) (118108,)


In [11]:


def get_categorical_cols(X):
    return X.select_dtypes(exclude=[np.number, 'bool']).columns.tolist()

cat_cols = get_categorical_cols(df[features])

# Build shared vocabulary, explicitly handling NaN -> placeholder string
category_maps = {}
for col in cat_cols:
    vals = df[col].astype(object).where(df[col].notna(), 'missing_value').astype(str).unique().tolist()
    category_maps[col] = vals

def cast_with_shared_categories(X, cat_cols, category_maps):
    X = X.copy()
    for col in cat_cols:
        X[col] = X[col].astype(object).where(X[col].notna(), 'missing_value').astype(str)
        X[col] = pd.Categorical(X[col], categories=category_maps[col])
    return X

val_split_idx = int(len(train_df) * 0.85)
train_sub = train_df.iloc[:val_split_idx]
val_sub = train_df.iloc[val_split_idx:]

X_train_sub = cast_with_shared_categories(train_sub[features], cat_cols, category_maps)
y_train_sub = train_sub['isFraud'].copy()
X_val_sub = cast_with_shared_categories(val_sub[features], cat_cols, category_maps)
y_val_sub = val_sub['isFraud'].copy()

X_train = cast_with_shared_categories(train_df[features], cat_cols, category_maps)
y_train = train_df['isFraud'].copy()
X_test = cast_with_shared_categories(test_df[features], cat_cols, category_maps)
y_test = test_df['isFraud'].copy()

print("X_train_sub:", X_train_sub.shape, "| X_val_sub:", X_val_sub.shape)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print(X_train_sub.dtypes.value_counts())

X_train_sub: (401567, 431) | X_val_sub: (70865, 431)
X_train: (472432, 431) | X_test: (118108, 431)
float64     397
category     12
category      4
category      1
int64         1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
category      1
Name: count, dtype: int64


In [11]:
print(df[features].dtypes.value_counts())

float64    399
str         31
int64        1
Name: count, dtype: int64


In [28]:
import sys
!"{sys.executable}" -m pip install optuna

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 17.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import optuna
print(optuna.__version__)

4.9.0


In [12]:
import xgboost as xgb
from sklearn.metrics import average_precision_score

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'scale_pos_weight': (len(y_train_sub) - y_train_sub.sum()) / y_train_sub.sum(),
        'eval_metric': 'aucpr',
        'enable_categorical': True,
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train_sub, y_train_sub)
    val_proba = model.predict_proba(X_val_sub)[:, 1]
    return average_precision_score(y_val_sub, val_proba)

In [15]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

[I 2026-07-24 02:50:26,677] A new study created in memory with name: no-name-b33abef5-eef6-4e73-9e1c-52c2605a68f2
d:\backup of old laptop\F Folder\Sir Tauseef\airllm_env\Lib\site-packages\xgboost\core.py:553: UserWarning: [02:50:36] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-07-24 02:50:36,360] Trial 0 finished with value: 0.4569170768201914 and parameters: {'n_estimators': 164, 'max_depth': 5, 'learning_rate': 0.012136722876533682, 'subsample': 0.9902958781665518, 'colsample_bytree': 0.6511196305720488, 'min_child_weight': 10}. Best is tri

In [16]:
print("Best AUPRC (validation):", study.best_value)
print("Best params:", study.best_params)

Best AUPRC (validation): 0.6203810526137208
Best params: {'n_estimators': 447, 'max_depth': 10, 'learning_rate': 0.16626905798769015, 'subsample': 0.9321135943451905, 'colsample_bytree': 0.8281518259712203, 'min_child_weight': 9}


In [17]:
best_params = study.best_params.copy()
best_params['scale_pos_weight'] = (len(y_train) - y_train.sum()) / y_train.sum()
best_params['eval_metric'] = 'aucpr'
best_params['enable_categorical'] = True
best_params['tree_method'] = 'hist'
best_params['device'] = 'cuda'
best_params['random_state'] = 42

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)



,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8281518259712203
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'aucpr'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [18]:
with open('models/best_params.json', 'r') as f:
    best_params = json.load(f)

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

print("Retrained final_model using saved best_params.")

Retrained final_model using saved best_params.


In [19]:
from sklearn.metrics import classification_report, average_precision_score, f1_score

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=4))
print("AUPRC:", average_precision_score(y_test, y_proba))
print("F1 (fraud class):", f1_score(y_test, y_pred))

              precision    recall  f1-score   support

           0     0.9828    0.9831    0.9830    114044
           1     0.5226    0.5180    0.5203      4064

    accuracy                         0.9671    118108
   macro avg     0.7527    0.7506    0.7516    118108
weighted avg     0.9670    0.9671    0.9671    118108

AUPRC: 0.5227540006589777
F1 (fraud class): 0.5202669303015324


In [20]:
import os
os.makedirs('models', exist_ok=True)

final_model.save_model('models/xgb_baseline.json')

In [21]:


import json
with open('models/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2, default=str)

print("Saved model and params.")

Saved model and params.


In [15]:
import shap
explainer = shap.TreeExplainer(final_model)

NameError: name 'final_model' is not defined

In [23]:
X_test_sample = X_test.sample(n=2000, random_state = 42)
shap_values_sample = explainer.shap_values(X_test_sample)
print(type(shap_values_sample), shap_values_sample.shape if hasattr(shap_values_sample, 'shape') else len(shap_values_sample))

<class 'numpy.ndarray'> (2000, 431)


In [24]:
from sklearn.model_selection import train_test_split

sample_size = 10000 # checking with 10k samples for SHAP values to avoid memory issues
X_test_shap, _, y_test_shap, _ = train_test_split(X_test, y_test, train_size = sample_size, stratify = y_test, random_state = 42)

print("X_test_shap shape:", X_test_shap.shape)
print("y_test_shap shape:", y_test_shap.shape)

X_test_shap shape: (10000, 431)
y_test_shap shape: (10000,)


In [25]:
import time
start = time.time()
shap_values_full = explainer.shap_values(X_test_shap)
elapsed_time = time.time() - start
print(f"Time Taken: {elapsed_time:.1f} seconds")
print(shap_values_full.shape)

Time Taken: 2.4 seconds
(10000, 431)


In [26]:
start = time.time()
shap_values_full = explainer.shap_values(X_test)
elapsed_time = time.time() - start
print(f"Time Taken: {elapsed_time:.1f} seconds")
print(shap_values_full.shape)

Time Taken: 27.9 seconds
(118108, 431)


In [14]:
def get_top_features(idx, X, shap_values, k = 5):
    """Return top k features by absolute SHAP values for a given row index"""
    contributions = shap_values[idx]
    top_idx = np.argsort(np.abs(contributions))[-k:][::-1]
    return [(X.columns[i], X.iloc[idx][X.columns[i]], contributions[i]) for i in top_idx]

fraud_indices = np.where(y_test.values == 1)[0]
sample_idx = fraud_indices[0]

top_features = get_top_features(sample_idx, X_test, shap_values_full, k = 5)
for feat_name, feat_val, shap_val in top_features:
    print(f"Feature: {feat_name}, Value: {feat_val}, SHAP value: {shap_val:.4f}")

NameError: name 'shap_values_full' is not defined

In [63]:
os.makedirs('outputs/shap', exist_ok=True)
np.save('outputs/shap/shap_values_full.npy', shap_values_full)
X_test.to_pickle('outputs/shap/X_test_full.pkl')
y_test.to_pickle('outputs/shap/y_test_full.pkl')
np.save('outputs/shap/expected_value.npy', np.array(explainer.expected_value))
print("Saved.")

Saved.


In [16]:
# ---Rerun---
shap_values_full = np.load('outputs/shap/shap_values_full.npy')
X_test = pd.read_pickle('outputs/shap/X_test_full.pkl')
y_test = pd.read_pickle('outputs/shap/y_test_full.pkl')
expected_value = np.load('outputs/shap/expected_value.npy')

print("Loaded SHAP outputs from disk. shap_values_full shape:", shap_values_full.shape)

Loaded SHAP outputs from disk. shap_values_full shape: (118108, 431)


In [17]:
# ---Ignore---
def get_top_features(idx, X, shap_values, k = 5):
    """Return top k features by absolute SHAP values for a given row index"""
    contributions = shap_values[idx]
    top_idx = np.argsort(np.abs(contributions))[-k:][::-1]
    return [(X.columns[i], X.iloc[idx][X.columns[i]], contributions[i]) for i in top_idx]

fraud_indices = np.where(y_test.values == 1)[0]
sample_idx = fraud_indices[0]

top_features = get_top_features(sample_idx, X_test, shap_values_full, k = 5)
for feat_name, feat_val, shap_val in top_features:
    print(f"Feature: {feat_name}, Value: {feat_val}, SHAP value: {shap_val:.4f}")

Feature: card2, Value: 103.0, SHAP value: -0.7997
Feature: id_31, Value: chrome 65.0, SHAP value: -0.6956
Feature: TransactionAmt, Value: 33.261, SHAP value: -0.6520
Feature: card1, Value: 9300, SHAP value: -0.5543
Feature: id_20, Value: 417.0, SHAP value: 0.4349


In [18]:
for i in fraud_indices[1:5]:
    print(f"\n--- Transaction index {i} ---")
    top_feats = get_top_features(i, X_test, shap_values_full, k=5)
    for feat_name, feat_val, shap_val in top_feats:
        print(f"{feat_name}: value={feat_val}, SHAP={shap_val:.4f}")


--- Transaction index 36 ---
TransactionAmt: value=49.0, SHAP=-0.6541
addr1: value=204.0, SHAP=-0.5376
M6: value=T, SHAP=-0.5033
card2: value=321.0, SHAP=-0.4587
P_emaildomain: value=gmail.com, SHAP=-0.3189

--- Transaction index 67 ---
C11: value=6.0, SHAP=0.8114
C8: value=8.0, SHAP=0.8080
V62: value=3.0, SHAP=0.7562
C1: value=4.0, SHAP=0.6753
C6: value=7.0, SHAP=0.5311

--- Transaction index 72 ---
V258: value=2.0, SHAP=1.1770
C11: value=6.0, SHAP=0.8723
C1: value=5.0, SHAP=0.8574
card1: value=16132, SHAP=-0.7904
C13: value=5.0, SHAP=-0.6437

--- Transaction index 75 ---
id_31: value=edge 16.0, SHAP=-1.3192
C14: value=0.0, SHAP=1.1412
C13: value=0.0, SHAP=0.9761
P_emaildomain: value=hotmail.com, SHAP=-0.7865
TransactionAmt: value=59.636, SHAP=-0.5443


In [19]:
nameable_set = set(nameable)
def count_nameable_in_top_k(idx, X, shap_values, k = 5):
    """Count how many of the top k features by absolute SHAP values are in the nameable set"""
    top_feats = get_top_features(idx, X, shap_values, k=k)
    count = sum(1 for feat_name, _, _ in top_feats if feat_name in nameable_set)
    return count
count = [count_nameable_in_top_k(i, X_test, shap_values_full, k=5) for i in fraud_indices]

counts_series = pd.Series(count)
print(counts_series.value_counts().sort_index())
print("\n Total fraud transactions:", len(counts_series))
print(f"% with 0 nameable features in top 5: {(counts_series == 0).mean() * 100:.1f}%")
print(f"% with 1-2 nameable features in top 5: {(counts_series.isin([1, 2])).mean() * 100:.1f}%")
print(f"% with 3+ nameable features in top 5: {(counts_series >= 3).mean() * 100:.1f}%")

0     691
1    1238
2    1362
3     630
4     121
5      22
Name: count, dtype: int64

 Total fraud transactions: 4064
% with 0 nameable features in top 5: 17.0%
% with 1-2 nameable features in top 5: 64.0%
% with 3+ nameable features in top 5: 19.0%


In [20]:
def interpretability_tier(count):
    if count == 0:
        return "low"       # 17% of cases
    elif count <= 2:
        return "medium"    # 64% of cases
    else:
        return "high"      # 19% of cases

tiers = counts_series.apply(interpretability_tier)
print(tiers.value_counts())

# Attach tier back to fraud transaction indices for later use in Phase 3/5
fraud_tier_map = dict(zip(fraud_indices, tiers))

medium    2600
high       773
low        691
Name: count, dtype: int64


In [21]:
def interpretability_tier(count):
    if count == 0:
        return "low"       # 17% of cases
    elif count <= 2:
        return "medium"    # 64% of cases
    else:
        return "high"      # 19% of cases

tiers = counts_series.apply(interpretability_tier)
print(tiers.value_counts())

# Attach tier back to fraud transaction indices for later use in Phase 3/5
fraud_tier_map = dict(zip(fraud_indices, tiers))

medium    2600
high       773
low        691
Name: count, dtype: int64


In [34]:
import pickle
with open('outputs/shap/fraud_tier_map.pkl', 'wb') as f:
    pickle.dump(fraud_tier_map, f)
print("Saved tier mapping.")

Saved tier mapping.


In [22]:
def describe_feature(feat_name, feat_val):
    """Turn a raw feature name+value into a human-readable phrase."""
    if feat_val in ('missing_value', 'nan') or pd.isna(feat_val):
        return f"a missing/undisclosed value for feature {feat_name}"
    if feat_name == 'TransactionAmt':
        try:
            return f"a transaction amount of ${float(feat_val):.2f}"
        except (ValueError, TypeError):
            return f"a transaction amount of {feat_val}"
    elif feat_name == 'ProductCD':
        return f"product category '{feat_val}'"
    elif feat_name == 'card4':
        return f"a {feat_val} card"
    elif feat_name == 'card6':
        return f"a {feat_val} card type"
    elif feat_name in ('card1', 'card2', 'card3', 'card5'):
        return f"an internal card identifier ({feat_name}) with value {feat_val}"
    elif feat_name == 'P_emaildomain':
        return f"purchaser email domain '{feat_val}'"
    elif feat_name == 'R_emaildomain':
        return f"recipient email domain '{feat_val}'"
    elif feat_name == 'DeviceType':
        return f"device type '{feat_val}'"
    elif feat_name == 'DeviceInfo':
        return f"device info '{feat_val}'"
    elif feat_name == 'addr1':
        return f"billing region code {feat_val}"
    elif feat_name == 'addr2':
        return f"billing country code {feat_val}"
    elif feat_name == 'dist1':
        return f"a distance metric of {feat_val}"
    elif feat_name == 'dist2':
        return f"a secondary distance metric of {feat_val}"
    elif feat_name.startswith('id_') and any(b in str(feat_val).lower() for b in ['chrome', 'firefox', 'safari', 'edge']):
        return f"browser/user-agent info '{feat_val}'"
    else:
        return f"an engineered risk indicator ({feat_name}) with value {feat_val}"

# Test it
test_feats = get_top_features(fraud_indices[0], X_test, shap_values_full, k=5)
for feat_name, feat_val, shap_val in test_feats:
    direction = "increasing" if shap_val > 0 else "decreasing"
    print(f"- {describe_feature(feat_name, feat_val)} ({direction} fraud likelihood)")

- an internal card identifier (card2) with value 103.0 (decreasing fraud likelihood)
- browser/user-agent info 'chrome 65.0' (decreasing fraud likelihood)
- a transaction amount of $33.26 (decreasing fraud likelihood)
- an internal card identifier (card1) with value 9300 (decreasing fraud likelihood)
- an engineered risk indicator (id_20) with value 417.0 (increasing fraud likelihood)


In [23]:
def generate_explanation(idx, X, shap_values, k=5):
    top_feats = get_top_features(idx, X, shap_values, k=k)
    
    fraud_indicators = []
    legit_indicators = []
    
    for feat_name, feat_val, shap_val in top_feats:
        desc = describe_feature(feat_name, feat_val)
        if shap_val > 0:
            fraud_indicators.append(desc)
        else:
            legit_indicators.append(desc)
    
    parts = []
    if fraud_indicators:
        parts.append(f"Factors increasing fraud risk: {'; '.join(fraud_indicators)}.")
    if legit_indicators:
        parts.append(f"Factors decreasing fraud risk: {'; '.join(legit_indicators)}.")
    
    return " ".join(parts)

# Test on the same fraud case
explanation = generate_explanation(fraud_indices[0], X_test, shap_values_full, k=5)
print(explanation)

Factors increasing fraud risk: an engineered risk indicator (id_20) with value 417.0. Factors decreasing fraud risk: an internal card identifier (card2) with value 103.0; browser/user-agent info 'chrome 65.0'; a transaction amount of $33.26; an internal card identifier (card1) with value 9300.


In [24]:
def generate_full_explanation_record(idx, X, y, shap_values, nameable_set, k=5):
    top_feats = get_top_features(idx, X, shap_values, k=k)
    nameable_count = sum(1 for feat_name, _, _ in top_feats if feat_name in nameable_set)
    tier = interpretability_tier(nameable_count)
    
    explanation_text = generate_explanation(idx, X, shap_values, k=k)
    true_label = "Fraud" if y.iloc[idx] == 1 else "Legitimate"
    
    return {
        'index': idx,
        'true_label': true_label,
        'explanation': explanation_text,
        'interpretability_tier': tier,
        'nameable_count': nameable_count
    }

# Test it
record = generate_full_explanation_record(fraud_indices[0], X_test, y_test, shap_values_full, nameable_set, k=5)
for key, val in record.items():
    print(f"{key}: {val}")

index: 0
true_label: Fraud
explanation: Factors increasing fraud risk: an engineered risk indicator (id_20) with value 417.0. Factors decreasing fraud risk: an internal card identifier (card2) with value 103.0; browser/user-agent info 'chrome 65.0'; a transaction amount of $33.26; an internal card identifier (card1) with value 9300.
interpretability_tier: medium
nameable_count: 1


In [25]:
def serialize_transaction(idx, X, nameable_set):
    """Turn a full transaction row into a natural-language description for the LLM prompt."""
    row = X.iloc[idx]
    
    parts = []
    for feat_name in nameable_set:
        if feat_name in X.columns:
            val = row[feat_name]
            if pd.notna(val) and str(val) not in ('nan', 'missing_value'):
                parts.append(describe_feature(feat_name, val))
    
    return "Transaction details: " + "; ".join(parts) + "."

sample_prompt = serialize_transaction(fraud_indices[0], X_test, nameable_set)
print(sample_prompt)

Transaction details: a transaction amount of $33.26; device type 'desktop'; a debit card type; purchaser email domain 'gmail.com'; recipient email domain 'gmail.com'; product category 'C'; a visa card.


In [31]:
def serialize_transaction_with_attributions(idx, X, y, shap_values, nameable_set, k=5):
    base_prompt = serialize_transaction(idx, X, nameable_set)
    top_feats = get_top_features(idx, X, shap_values, k=k)
    
    attribution_lines = []
    for feat_name, feat_val, shap_val in top_feats:
        direction = "increases" if shap_val > 0 else "decreases"
        if feat_val in ('missing_value', 'nan') or pd.isna(feat_val):
            display_val = "missing/undisclosed"
        else:
            display_val = feat_val
        attribution_lines.append(f"{feat_name} (value: {display_val}) {direction} fraud risk")
    
    attributions_text = "Model-identified risk factors: " + "; ".join(attribution_lines) + "."
    return base_prompt + " " + attributions_text

# Re-test
example = build_training_example(fraud_indices[0], X_test, y_test, shap_values_full, nameable_set)
for key, val in example.items():
    print(f"{key}: {val}\n")

prompt: Transaction details: a transaction amount of $33.26; device type 'desktop'; a debit card type; purchaser email domain 'gmail.com'; recipient email domain 'gmail.com'; product category 'C'; a visa card. Model-identified risk factors: card2 (value: 103.0) decreases fraud risk; id_31 (value: chrome 65.0) decreases fraud risk; TransactionAmt (value: 33.261) decreases fraud risk; card1 (value: 9300) decreases fraud risk; id_20 (value: 417.0) increases fraud risk.

true_label: Fraud

target_explanation: Factors increasing fraud risk: an engineered risk indicator (id_20) with value 417.0. Factors decreasing fraud risk: an internal card identifier (card2) with value 103.0; browser/user-agent info 'chrome 65.0'; a transaction amount of $33.26; an internal card identifier (card1) with value 9300.

interpretability_tier: medium



In [32]:
def build_training_example(idx, X, y, shap_values, nameable_set, k=5):
    prompt = serialize_transaction_with_attributions(idx, X, y, shap_values, nameable_set, k=k)
    record = generate_full_explanation_record(idx, X, y, shap_values, nameable_set, k=k)
    return {
        'prompt': prompt,
        'true_label': record['true_label'],
        'target_explanation': record['explanation'],
        'interpretability_tier': record['interpretability_tier']
    }

# Test it
example = build_training_example(fraud_indices[0], X_test, y_test, shap_values_full, nameable_set)
for key, val in example.items():
    print(f"{key}: {val}\n")

prompt: Transaction details: a transaction amount of $33.26; device type 'desktop'; a debit card type; purchaser email domain 'gmail.com'; recipient email domain 'gmail.com'; product category 'C'; a visa card. Model-identified risk factors: card2 (value: 103.0) decreases fraud risk; id_31 (value: chrome 65.0) decreases fraud risk; TransactionAmt (value: 33.261) decreases fraud risk; card1 (value: 9300) decreases fraud risk; id_20 (value: 417.0) increases fraud risk.

true_label: Fraud

target_explanation: Factors increasing fraud risk: an engineered risk indicator (id_20) with value 417.0. Factors decreasing fraud risk: an internal card identifier (card2) with value 103.0; browser/user-agent info 'chrome 65.0'; a transaction amount of $33.26; an internal card identifier (card1) with value 9300.

interpretability_tier: medium



In [33]:
random.seed(42)
legit_indices = np.where(y_test.values == 0)[0]
legit_idx_sample = random.sample(list(legit_indices), len(fraud_indices))
all_indices = list(fraud_indices) + legit_idx_sample
random.shuffle(all_indices)

In [34]:
import time
training_data = []
start = time.time()
for i, idx in enumerate(all_indices):
    ex = build_training_example(idx, X_test, y_test, shap_values_full, nameable_set)
    ex['index'] = int(idx)
    training_data.append(ex)
    if (i+1) % 1000 == 0:
        print(f"Processed {i+1}/{len(all_indices)}. Time elapsed: {time.time()-start:.1f}s")

print(f"Done. Total: {len(training_data)}. Time: {time.time()-start:.1f}s")

Processed 1000/8128. Time elapsed: 11.0s
Processed 2000/8128. Time elapsed: 24.0s
Processed 3000/8128. Time elapsed: 36.6s
Processed 4000/8128. Time elapsed: 48.7s
Processed 5000/8128. Time elapsed: 60.4s
Processed 6000/8128. Time elapsed: 71.9s
Processed 7000/8128. Time elapsed: 85.9s
Processed 8000/8128. Time elapsed: 98.8s
Done. Total: 8128. Time: 101.0s


In [35]:
with open('ieee-fraud-detection/llm_training_data.json', 'w') as f:
    json.dump(training_data, f, indent = 2)
print("Saved training data to 'ieee-fraud-detection/llm_training_data.json'.")
leak_count = sum(1 for ex in training_data if 'missing_value' in ex['target_explanation'] or 'missing_value' in ex['prompt'])
print(f"Remaining 'missing_value' leaks: {leak_count}")
print("Saved corrected training data.")



Saved training data to 'ieee-fraud-detection/llm_training_data.json'.
Remaining 'missing_value' leaks: 0
Saved corrected training data.


In [36]:
labels = [ex['true_label'] for ex in training_data]
tiers = [ex['interpretability_tier'] for ex in training_data]
print(pd.Series(labels).value_counts())
print(pd.Series(tiers).value_counts())

Legitimate    4064
Fraud         4064
Name: count, dtype: int64
medium    5373
high      1992
low        763
Name: count, dtype: int64


In [37]:
random.seed(1)
sample_check = random.sample(training_data, 4)

for ex in sample_check:
    print(f"[{ex['true_label']} | {ex['interpretability_tier']}]")
    print(f"Prompt: {ex['prompt']}")
    print(f"Explanation : {ex['target_explanation']}\n")

[Fraud | medium]
Prompt: Transaction details: a transaction amount of $87.00; a distance metric of 753.0; a debit card type; billing country code 87.0; purchaser email domain 'gmail.com'; product category 'W'; a visa card; billing region code 325.0. Model-identified risk factors: addr1 (value: 325.0) decreases fraud risk; M6 (value: missing/undisclosed) decreases fraud risk; D2 (value: 1.0) increases fraud risk; C13 (value: 1.0) increases fraud risk; TransactionAmt (value: 87.0) decreases fraud risk.
Explanation : Factors increasing fraud risk: an engineered risk indicator (D2) with value 1.0; an engineered risk indicator (C13) with value 1.0. Factors decreasing fraud risk: billing region code 325.0; a missing/undisclosed value for feature M6; a transaction amount of $87.00.

[Legitimate | high]
Prompt: Transaction details: a transaction amount of $34.00; a debit card type; billing country code 87.0; purchaser email domain 'gmail.com'; product category 'W'; a visa card; billing region 

In [38]:
def format_for_mistral(example):
     instruction = (
        "You are a fraud detection assistant. Given the transaction details below, "
        "classify the transaction as Fraud or Legitimate, and explain your reasoning "
        "based on the key risk factors."
    )
     user_content = f"{instruction} \n\n{example['prompt']}"
     assistant_content = f"Classification: {example['true_label']}\n{example['target_explanation']}"

     return {
          "text": f"<s>[INST] {user_content} [/INST] {assistant_content}</s>"
     }

formatted_data = [format_for_mistral(ex) for ex in training_data]
print(formatted_data[0]['text'])

<s>[INST] You are a fraud detection assistant. Given the transaction details below, classify the transaction as Fraud or Legitimate, and explain your reasoning based on the key risk factors. 

Transaction details: a transaction amount of $49.00; a debit card type; billing country code 87.0; product category 'W'; a visa card; billing region code 272.0. Model-identified risk factors: addr1 (value: 272.0) decreases fraud risk; C13 (value: 21.0) decreases fraud risk; D15 (value: 90.0) decreases fraud risk; card1 (value: 4436) decreases fraud risk; V285 (value: 10.0) decreases fraud risk. [/INST] Classification: Legitimate
Factors decreasing fraud risk: billing region code 272.0; an engineered risk indicator (C13) with value 21.0; an engineered risk indicator (D15) with value 90.0; an internal card identifier (card1) with value 4436; an engineered risk indicator (V285) with value 10.0.</s>


In [39]:
from sklearn.model_selection import train_test_split
train_formatted, val_formatted = train_test_split(formatted_data, test_size = 0.1, random_state = 42)
print(f"Fine-tuning train size: {len(train_formatted)}")
print(f"Fine-tuning val size: {len(val_formatted)}")

Fine-tuning train size: 7315
Fine-tuning val size: 813


In [40]:
import json

with open('ieee-fraud-detection/llm_train_formatted.json', 'w') as f:
    json.dump(train_formatted, f, indent=2)

with open('ieee-fraud-detection/llm_val_formatted.json', 'w') as f:
    json.dump(val_formatted, f, indent=2)

print("Saved formatted train/val sets.")

Saved formatted train/val sets.


In [41]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6418adf5d459c9e7fbad92a5', 'name': 'aazoo', 'fullname': 'Chhapra', 'email': 'aazhmeerchhapra@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1785542400, 'isPro': False, 'avatarUrl': '/avatars/ad80f292f34bdddc4bb779ca252781e2.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'test2', 'role': 'write', 'createdAt': '2026-06-02T23:46:15.441Z'}}}


In [42]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [43]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

In [44]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto")
print("Loaded model and tokenizer.")
print(f"Memory footprint (GB): {model.get_memory_footprint() / 1e9:.2f}")

W0724 04:22:55.416000 21640 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

d:\backup of old laptop\F Folder\Sir Tauseef\airllm_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Loaded model and tokenizer.
Memory footprint (GB): 4.01


In [45]:
import sys
!"{sys.executable}" -m pip install "transformers==4.46.3"

   ---------------------------------------- 0.0/10.0 MB ? eta -:--:--
   --------------- ------------------------ 3.9/10.0 MB 26.0 MB/s eta 0:00:01
   ---------------------------------------  10.0/10.0 MB 28.2 MB/s eta 0:00:01
   ---------------------------------------- 10.0/10.0 MB 26.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 26.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.43.4
    Uninstalling transformers-4.43.4:
      Successfully uninstalled transformers-4.43.4


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optimum 1.21.0 requires numpy<2.0, but you have numpy 2.4.6 which is incompatible.
optimum 1.21.0 requires transformers[sentencepiece]<4.43.0,>=4.26.0, but you have transformers 4.46.3 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [45]:
import json

with open('ieee-fraud-detection/llm_train_formatted.json', 'r') as f:
    train_formatted = json.load(f)
with open('ieee-fraud-detection/llm_val_formatted.json', 'r') as f:
    val_formatted = json.load(f)

print(len(train_formatted), len(val_formatted))

7315 813


In [46]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model type after LoRA:", type(model))

trainable params: 83,886,080 || all params: 7,325,618,176 || trainable%: 1.1451
Model type after LoRA: <class 'peft.peft_model.PeftModelForCausalLM'>


In [9]:
import sys
!"{sys.executable}" -m pip install peft

   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   --------------------------------------- 680.7/680.7 kB 13.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 25.8 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 82.0.1
    Uninstalling setuptools-82.0.1:
      Successfully uninstalled setuptools-82.0.1



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

MAX_LEN = 320

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"], truncation=True, max_length=MAX_LEN,
        padding="max_length", add_special_tokens=False
    )
    labels = []
    for i, ids in enumerate(tokenized["input_ids"]):
        text = examples["text"][i]
        unpadded_ids = tokenizer(text, truncation=True, max_length=MAX_LEN, add_special_tokens=False)["input_ids"]
        real_len = len(unpadded_ids)

        prompt_part = text.split("[/INST]")[0] + "[/INST]"
        prompt_token_len = len(tokenizer(prompt_part, add_special_tokens=False)["input_ids"])

        label = list(ids)
        for j in range(len(label)):
            if j < prompt_token_len or j >= real_len:
                label[j] = -100
        labels.append(label)
    tokenized["labels"] = labels
    return tokenized

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print("Re-tokenized with EOS fix.", train_tokenized)

Map:   0%|          | 0/7315 [00:00<?, ? examples/s]

Map:   0%|          | 0/813 [00:00<?, ? examples/s]

Re-tokenized with EOS fix. Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 7315
})


In [48]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./models/mistral_fraud_lora",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    warmup_ratio=0.03,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
print("Trainer configured.")

Trainer configured.


In [49]:
tiny_train = train_tokenized.select(range(20))
tiny_val = val_tokenized.select(range(10))

trainer_test = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./models/test_run",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=1,
        logging_steps=1,
        bf16=True,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        report_to="none",
    ),
    train_dataset=tiny_train,
    eval_dataset=tiny_val,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer_test.train()

  0%|          | 0/5 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


{'loss': 2.7723, 'grad_norm': 22.710590362548828, 'learning_rate': 4e-05, 'epoch': 0.2}
{'loss': 2.0876, 'grad_norm': 13.089690208435059, 'learning_rate': 3e-05, 'epoch': 0.4}
{'loss': 1.7211, 'grad_norm': 10.106195449829102, 'learning_rate': 2e-05, 'epoch': 0.6}
{'loss': 1.4799, 'grad_norm': 9.211548805236816, 'learning_rate': 1e-05, 'epoch': 0.8}
{'loss': 1.2525, 'grad_norm': 9.570387840270996, 'learning_rate': 0.0, 'epoch': 1.0}
{'train_runtime': 22.8512, 'train_samples_per_second': 0.875, 'train_steps_per_second': 0.219, 'train_loss': 1.8626888990402222, 'epoch': 1.0}


TrainOutput(global_step=5, training_loss=1.8626888990402222, metrics={'train_runtime': 22.8512, 'train_samples_per_second': 0.875, 'train_steps_per_second': 0.219, 'total_flos': 276270573158400.0, 'train_loss': 1.8626888990402222, 'epoch': 1.0})

In [50]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print(f"Max allocated during run: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

Allocated: 5.01 GB
Reserved: 6.73 GB
Max allocated during run: 6.41 GB


In [51]:
trainer.train()

  0%|          | 0/1371 [00:00<?, ?it/s]

{'loss': 0.8405, 'grad_norm': 19.32816505432129, 'learning_rate': 4.761904761904762e-05, 'epoch': 0.02}
{'loss': 0.3319, 'grad_norm': 7.44509220123291, 'learning_rate': 9.523809523809524e-05, 'epoch': 0.04}
{'loss': 0.2652, 'grad_norm': 13.828134536743164, 'learning_rate': 0.00014285714285714287, 'epoch': 0.07}
{'loss': 0.2191, 'grad_norm': 4.325806140899658, 'learning_rate': 0.00019047619047619048, 'epoch': 0.09}
{'loss': 0.1992, 'grad_norm': 3.4433295726776123, 'learning_rate': 0.00019879608728367193, 'epoch': 0.11}


  0%|          | 0/407 [00:00<?, ?it/s]

{'eval_loss': 0.1931987851858139, 'eval_runtime': 239.8299, 'eval_samples_per_second': 3.39, 'eval_steps_per_second': 1.697, 'epoch': 0.11}
{'loss': 0.1884, 'grad_norm': 2.862424612045288, 'learning_rate': 0.00019729119638826185, 'epoch': 0.13}
{'loss': 0.1788, 'grad_norm': 2.762430429458618, 'learning_rate': 0.00019578630549285179, 'epoch': 0.15}
{'loss': 0.1794, 'grad_norm': 3.1648569107055664, 'learning_rate': 0.0001942814145974417, 'epoch': 0.17}
{'loss': 0.172, 'grad_norm': 2.5842928886413574, 'learning_rate': 0.0001927765237020316, 'epoch': 0.2}
{'loss': 0.1784, 'grad_norm': 3.0278124809265137, 'learning_rate': 0.00019127163280662152, 'epoch': 0.22}


  0%|          | 0/407 [00:00<?, ?it/s]

...

  0%|          | 0/407 [00:00<?, ?it/s]

{'eval_loss': 0.12324094027280807, 'eval_runtime': 238.5857, 'eval_samples_per_second': 3.408, 'eval_steps_per_second': 1.706, 'epoch': 2.84}
{'loss': 0.1081, 'grad_norm': 1.707834243774414, 'learning_rate': 9.179834462001505e-06, 'epoch': 2.86}
{'loss': 0.1081, 'grad_norm': 1.815974473953247, 'learning_rate': 7.674943566591422e-06, 'epoch': 2.89}
{'loss': 0.1109, 'grad_norm': 1.669257402420044, 'learning_rate': 6.170052671181339e-06, 'epoch': 2.91}
{'loss': 0.1103, 'grad_norm': 2.038882255554199, 'learning_rate': 4.665161775771257e-06, 'epoch': 2.93}
{'loss': 0.1075, 'grad_norm': 1.6862282752990723, 'learning_rate': 3.1602708803611743e-06, 'epoch': 2.95}


  0%|          | 0/407 [00:00<?, ?it/s]

{'eval_loss': 0.1226544976234436, 'eval_runtime': 238.5956, 'eval_samples_per_second': 3.407, 'eval_steps_per_second': 1.706, 'epoch': 2.95}
{'loss': 0.1017, 'grad_norm': 2.4116854667663574, 'learning_rate': 1.655379984951091e-06, 'epoch': 2.97}
{'loss': 0.1116, 'grad_norm': 1.8572405576705933, 'learning_rate': 1.5048908954100828e-07, 'epoch': 3.0}
{'train_runtime': 27026.3674, 'train_samples_per_second': 0.812, 'train_steps_per_second': 0.051, 'train_loss': 0.14485397501247457, 'epoch': 3.0}


TrainOutput(global_step=1371, training_loss=0.14485397501247457, metrics={'train_runtime': 27026.3674, 'train_samples_per_second': 0.812, 'train_steps_per_second': 0.051, 'total_flos': 3.029859375828173e+17, 'train_loss': 0.14485397501247457, 'epoch': 2.9983597594313833})

In [52]:
log_df = pd.DataFrame(trainer.state.log_history)


In [53]:
log_df.head()

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,0.8405,19.328165,0.000048,0.021870,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.3319,7.445092,0.000095,0.043740,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.2652,13.828135,0.000143,0.065610,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.2191,4.325806,0.000190,0.087479,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.1992,3.443330,0.000199,0.109349,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [54]:
eval_only = log_df[log_df['eval_loss'].notna()][['step', 'eval_loss', 'epoch']]
print(eval_only)

     step  eval_loss     epoch
5      50   0.193199  0.109349
11    100   0.174780  0.218699
17    150   0.165655  0.328048
23    200   0.159388  0.437397
29    250   0.158940  0.546747
35    300   0.153596  0.656096
41    350   0.151427  0.765446
47    400   0.147883  0.874795
53    450   0.144681  0.984144
59    500   0.144437  1.093494
65    550   0.142496  1.202843
71    600   0.141033  1.312192
77    650   0.138376  1.421542
83    700   0.137252  1.530891
89    750   0.134838  1.640241
95    800   0.133856  1.749590
101   850   0.132381  1.858939
107   900   0.130152  1.968289
113   950   0.130374  2.077638
119  1000   0.129632  2.186987
125  1050   0.127936  2.296337
131  1100   0.127242  2.405686
137  1150   0.126499  2.515036
143  1200   0.124319  2.624385
149  1250   0.123565  2.733734
155  1300   0.123241  2.843084
161  1350   0.122654  2.952433


In [15]:
eval_only = log_df[log_df['eval_loss'].notna()][['step', 'eval_loss', 'epoch']]
print(eval_only)

     step  eval_loss     epoch
5      50   0.278278  0.109349
11    100   0.243353  0.218699
17    150   0.230387  0.328048
23    200   0.224411  0.437397
29    250   0.219383  0.546747
35    300   0.217622  0.656096
41    350   0.209240  0.765446
47    400   0.208560  0.874795
53    450   0.204183  0.984144
59    500   0.200926  1.093494
65    550   0.199383  1.202843
71    600   0.197774  1.312192
77    650   0.193785  1.421542
83    700   0.192054  1.530891
89    750   0.189332  1.640241
95    800   0.188965  1.749590
101   850   0.184988  1.858939
107   900   0.181598  1.968289
113   950   0.182439  2.077638
119  1000   0.181879  2.186987
125  1050   0.180123  2.296337
131  1100   0.178975  2.405686
137  1150   0.176854  2.515036
143  1200   0.174495  2.624385
149  1250   0.173348  2.733734
155  1300   0.172319  2.843084
161  1350   0.171190  2.952433


In [57]:
os.makedirs('models/mistral_fraud_lora_v2', exist_ok=True)
model.save_pretrained('models/mistral_fraud_lora_v2')
tokenizer.save_pretrained('models/mistral_fraud_lora_v2')
print('Saved v2 LoRA adapter and tokenizer')

Saved v2 LoRA adapter and tokenizer


In [4]:
shap_values_full = np.load('outputs/shap/shap_values_full.npy')
X_test = pd.read_pickle('outputs/shap/X_test_full.pkl')
y_test = pd.read_pickle('outputs/shap/y_test_full.pkl')

with open('ieee-fraud-detection/llm_val_formatted.json', 'r') as f:
    val_formatted = json.load(f)

all_cols = list(X_test.columns)

print("Data loaded.", shap_values_full.shape, len(val_formatted))

Data loaded. (118108, 431) 813


In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, "models/mistral_fraud_lora_v2")
print("Fine-tuned model loaded.", type(model))

W0726 01:04:41.453000 7048 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

d:\backup of old laptop\F Folder\Sir Tauseef\airllm_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Fine-tuned model loaded. <class 'peft.peft_model.PeftModelForCausalLM'>


In [6]:
def generate_response(prompt_text, max_new_tokens=150):
    full_prompt = f"<s>[INST] You are a fraud detecxtion assistant. Given the transaction details below, classify the transaction as Fraud or Legitimate, and explain your reasoning based on the key risk factors. \n\n{prompt_text} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors='pt', add_special_tokens=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False,
            pad_token_id = tokenizer.eos_token_id
        )
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated
    


for sample in val_formatted[:5]:
    full_text = sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = prompt_section.split('Transaction details:')[-1]
    prompt_for_generation = "Transaction details:" + transaction_part

    ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()
    generated = generate_response(prompt_for_generation)

    print("Prompt:", prompt_for_generation.strip())
    print("\nGROUND TRUTH:", ground_truth)
    print("\nMODEL GENERATED:", generated)
    print("\n" + "="*80 + "\n")


d:\backup of old laptop\F Folder\Sir Tauseef\airllm_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Prompt: Transaction details: a transaction amount of $44.50; a debit card type; billing country code 87.0; purchaser email domain 'yahoo.com'; product category 'W'; a mastercard card; billing region code 324.0. Model-identified risk factors: addr1 (value: 324.0) decreases fraud risk; card2 (value: 360.0) decreases fraud risk; card6 (value: debit) decreases fraud risk; TransactionAmt (value: 44.5) decreases fraud risk; card5 (value: 229.0) decreases fraud risk.

GROUND TRUTH: Classification: Legitimate
Factors decreasing fraud risk: billing region code 324.0; an internal card identifier (card2) with value 360.0; a debit card type; a transaction amount of $44.50; an internal card identifier (card5) with value 229.0.

MODEL GENERATED: Classification: Legitimate
Factors decreasing fraud risk: billing region code 324.0; an internal card identifier (card2) with value 360.0; a debit card type; a transaction amount of $44.50; an internal card identifier (card5) with value 229.0. Factors increa

In [7]:
def generate_response(prompt_text, max_new_tokens=150):
    full_prompt = f"<s>[INST] You are a fraud detection assistant. Given the transaction details below, classify the transaction as Fraud or Legitimate, and explain your reasoning based on the key risk factors.\n\n{prompt_text} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

# Re-run on the same 5 examples to compare
for sample in val_formatted[:5]:
    full_text = sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = prompt_section.split("Transaction details:")[-1]
    prompt_for_generation = "Transaction details:" + transaction_part
    ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()
    generated = generate_response(prompt_for_generation)
    print("GROUND TRUTH:", ground_truth)
    print("\nMODEL GENERATED:", generated)
    print("\n" + "="*80 + "\n")

GROUND TRUTH: Classification: Legitimate
Factors decreasing fraud risk: billing region code 324.0; an internal card identifier (card2) with value 360.0; a debit card type; a transaction amount of $44.50; an internal card identifier (card5) with value 229.0.

MODEL GENERATED: Classification: Legitimate
Factors decreasing fraud risk: billing regioncode 32 four.0; an internal card identifier (card2) with value 36o.0; a debt card type; a transaction amount o f$44. 50; an interior card identifier (car5) with value  229 . 0. Factors increasing fraud risk: M5 (value : T) increases fraud risk; C13 (value : 1.0) increases fraud r isk; V294 (value : M1) increases fraud risks; P_emaildomain (value: yahoo. com) decreases frau d risk; D15 (


GROUND TRUTH: Classification: Fraud
Factors increasing fraud risk: an engineered risk indicator (C9) with value 6.0; an engineered risk indicator (V76) with value 3.0; billing region code 472.0; an engineered risk indicator (M5) with value T. Factors decreasin

In [8]:
def generate_response(prompt_text, max_new_tokens=100):
    full_prompt = f"<s>[INST] You are a fraud detection assistant. Given the transaction details below, classify the transaction as Fraud or Legitimate, and explain your reasoning based on the key risk factors.\n\n{prompt_text} [/INST]"
    inputs = tokenizer(full_prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

for sample in val_formatted[:5]:
    full_text = sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = prompt_section.split("Transaction details:")[-1]
    prompt_for_generation = "Transaction details:" + transaction_part
    ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()
    generated = generate_response(prompt_for_generation)
    print("GROUND TRUTH:", ground_truth)
    print("\nMODEL GENERATED:", generated)
    print("\n" + "="*80 + "\n")

GROUND TRUTH: Classification: Legitimate
Factors decreasing fraud risk: billing region code 324.0; an internal card identifier (card2) with value 360.0; a debit card type; a transaction amount of $44.50; an internal card identifier (card5) with value 229.0.

MODEL GENERATED: Classification: Legitimate
Factors decreasing fraud risk: billing region code 324.0; an internal card identifier (card2) with value 360.0; a debit card type; a transaction amount of $44.50; an internal card identifier (card5) with value 229.0. Factors increasing fraud risk: M5 (value: T) increases fraud risk; card1 (value: 


GROUND TRUTH: Classification: Fraud
Factors increasing fraud risk: an engineered risk indicator (C9) with value 6.0; an engineered risk indicator (V76) with value 3.0; billing region code 472.0; an engineered risk indicator (M5) with value T. Factors decreasing fraud risk: an engineered risk indicator (D2) with value 237.0.

MODEL GENERATED: Classification: Fraud
Factors increasing fraud risk:

In [64]:
def evaluate_generation(sample):
    full_text = sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = prompt_section.split("Transaction details:")[-1]
    prompt_for_generation = "Transaction details:" + transaction_part
    ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()
    generated = generate_response(prompt_for_generation)
    
    gt_label = ground_truth.split("\n")[0].replace("Classification:", "").strip()
    gen_label = generated.split("\n")[0].replace("Classification:", "").strip() if "Classification:" in generated else "UNKNOWN"
    
    correct_class = (gt_label == gen_label)
    clean_stop = generated.count("Factors increasing") <= 1 and generated.count("Factors decreasing") <= 1
    
    return {'correct_classification': correct_class, 'clean_stop': clean_stop, 'generated': generated}

results = [evaluate_generation(s) for s in val_formatted[:30]]
import pandas as pd
results_df = pd.DataFrame(results)
print(f"Classification accuracy: {results_df['correct_classification'].mean()*100:.1f}%")
print(f"Clean stop rate: {results_df['clean_stop'].mean()*100:.1f}%")

Classification accuracy: 86.7%
Clean stop rate: 93.3%


In [65]:
for i, (sample, r) in enumerate(zip(val_formatted[:30], results)):
    if not r['correct_classification']:
        full_text = sample['text']
        prompt_section = full_text.split("[/INST]")[0]
        ground_truth = full_text.split("[/INST]")[1].replace("</s>", "").strip()
        print(f"--- Misclassified example {i} ---")
        print("PROMPT (risk factors):", prompt_section.split("Model-identified risk factors:")[-1].strip())
        print("GROUND TRUTH:", ground_truth[:100])
        print("GENERATED:", r['generated'][:150])
        print()

--- Misclassified example 2 ---
PROMPT (risk factors): addr1 (value: 325.0) decreases fraud risk; D2 (value: 1.0) increases fraud risk; card2 (value: 361.0) decreases fraud risk; M9 (value: F) decreases fraud risk; card6 (value: credit) increases fraud risk.
GROUND TRUTH: Classification: Legitimate
Factors increasing fraud risk: an engineered risk indicator (D2) with val
GENERATED: Classification: Fraud
Factors increasing fraud risk: an engineered risk indicator (D2) with value 1.0; a credit card type. Factors decreasing fraud ri

--- Misclassified example 6 ---
PROMPT (risk factors): addr1 (value: 512.0) decreases fraud risk; C1 (value: 8.0) increases fraud risk; card1 (value: 2753) decreases fraud risk; C13 (value: 11.0) decreases fraud risk; V70 (value: 1.0) decreases fraud risk.
GROUND TRUTH: Classification: Fraud
Factors increasing fraud risk: an engineered risk indicator (C1) with value 8.
GENERATED: Classification: Legitimate
Factors increasing fraud risk: an engineered risk ind

In [66]:
# Does increasing k reduce this specific error mode? Quick check on just these 4 cases
misclassified_indices = [2, 6, 12, 24]
for i in misclassified_indices:
    orig_idx = val_formatted[i]  # need original index if tracked - check if you have it
    print(f"Example {i} would need k=10 check - see if additional features change the picture")

Example 2 would need k=10 check - see if additional features change the picture
Example 6 would need k=10 check - see if additional features change the picture
Example 12 would need k=10 check - see if additional features change the picture
Example 24 would need k=10 check - see if additional features change the picture


In [67]:
# Rebuild a lookup from prompt text -> original index, using training_data (which still has 'index')
prompt_to_index = {}
for ex in training_data:
    # reconstruct the prompt exactly as it appears in formatted text
    key = ex['prompt'].split("Model-identified risk factors:")[0].strip()
    prompt_to_index[key] = ex['index']

def get_original_index(val_sample):
    full_text = val_sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = "Transaction details:" + prompt_section.split("Transaction details:")[-1]
    base_prompt = transaction_part.split("Model-identified risk factors:")[0].strip()
    return prompt_to_index.get(base_prompt, None)

# Now check k=10 for the 4 misclassified cases
misclassified_positions = [2, 6, 12, 24]

for pos in misclassified_positions:
    orig_idx = get_original_index(val_formatted[pos])
    if orig_idx is None:
        print(f"Position {pos}: could not match back to original index")
        continue
    
    top10 = get_top_features(orig_idx, X_test, shap_values_full, k=10)
    true_label = "Fraud" if y_test.iloc[orig_idx] == 1 else "Legitimate"
    print(f"\n--- Position {pos} (original index {orig_idx}) | True label: {true_label} ---")
    for feat_name, feat_val, shap_val in top10:
        direction = "increases" if shap_val > 0 else "decreases"
        print(f"  {feat_name}: {feat_val} ({direction} fraud risk, SHAP={shap_val:.3f})")


--- Position 2 (original index 100582) | True label: Legitimate ---
  addr1: 325.0 (decreases fraud risk, SHAP=-1.607)
  D2: 1.0 (increases fraud risk, SHAP=0.726)
  card2: 361.0 (decreases fraud risk, SHAP=-0.615)
  M9: F (decreases fraud risk, SHAP=-0.574)
  card6: credit (increases fraud risk, SHAP=0.557)
  C13: 1.0 (increases fraud risk, SHAP=0.522)
  card1: 4121 (decreases fraud risk, SHAP=-0.423)
  M6: F (increases fraud risk, SHAP=0.308)
  V13: 2.0 (increases fraud risk, SHAP=0.301)
  V281: 1.0 (increases fraud risk, SHAP=0.286)

--- Position 6 (original index 64427) | True label: Fraud ---
  addr1: 512.0 (decreases fraud risk, SHAP=-1.846)
  C1: 8.0 (increases fraud risk, SHAP=1.102)
  card1: 2753 (decreases fraud risk, SHAP=-1.057)
  C13: 11.0 (decreases fraud risk, SHAP=-0.871)
  V70: 1.0 (decreases fraud risk, SHAP=-0.828)
  C9: 6.0 (increases fraud risk, SHAP=0.711)
  C2: 8.0 (increases fraud risk, SHAP=0.678)
  TransactionAmt: 107.95 (decreases fraud risk, SHAP=-0.613)
  

In [9]:
import re

def extract_mentioned_features(generated_text, all_feature_names):
    mentioned = []
    for feat_name in all_feature_names:
        pattern = r'\b' + re.escape(feat_name) + r'\b'
        if re.search(pattern, generated_text):
            mentioned.append(feat_name)
    return mentioned

test_text = "Factors increasing fraud risk: C9 (value: 6.0) increases fraud risk; V76 (value: 3.0) increases fraud risk."
all_cols = list(X_test.columns)
mentioned = extract_mentioned_features(test_text, all_cols)
print(mentioned)

['C9', 'V76']


In [10]:
def compute_faithfullness_overlap(sample, all_feature_names, k=5):
    full_text = sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = "Transaction details:" + prompt_section.split("Transaction details:")[-1]

    given_features = extract_mentioned_features(transaction_part, all_feature_names)
    generated = generate_response(transaction_part)

    mentioned_features = extract_mentioned_features(generated, all_feature_names)

    if len(given_features) == 0:
        return None

    correctly_mentioned = set(mentioned_features) & set(given_features)
    fabricated = set(mentioned_features) - set(given_features)

    return{
        'given_count': len(given_features),
        'mentioned_count': len(mentioned_features),
        'correct_overlap': len(correctly_mentioned),
        'fabricated_count': len(fabricated),
        'recall': len(correctly_mentioned) / len(given_features) if given_features else None,
        'fabricated_features': list(fabricated),
    }


In [11]:
result = compute_faithfullness_overlap(val_formatted[0], all_cols)
print(result)

{'given_count': 5, 'mentioned_count': 4, 'correct_overlap': 2, 'fabricated_count': 2, 'recall': 0.4, 'fabricated_features': ['M5', 'card1']}


In [117]:
full_text = val_formatted[0]['text']
prompt_section = full_text.split("[/INST]")[0]
transaction_part = "Transaction details:" + prompt_section.split("Transaction details:")[-1]

generated = generate_response(transaction_part)
print("PROMPT:", transaction_part)
print("\nGENERATED:", generated)

PROMPT: Transaction details: a transaction amount of $44.50; a debit card type; billing country code 87.0; purchaser email domain 'yahoo.com'; product category 'W'; a mastercard card; billing region code 324.0. Model-identified risk factors: addr1 (value: 324.0) decreases fraud risk; card2 (value: 360.0) decreases fraud risk; card6 (value: debit) decreases fraud risk; TransactionAmt (value: 44.5) decreases fraud risk; card5 (value: 229.0) decreases fraud risk. 

GENERATED: Classification: Legitimate
Factors decreasing fraud risk: billing region code 324.0; an internal card identifier (card2) with value 360.0; a debit card type; a transaction amount of $44.50; an internal card identifier (card5) with value 229.0. Factors increasing fraud risk: M5 (value: T) increases fraud risk; card1 (value: 


In [12]:
def get_primary_response(generated_text):
    """Cut off at the first repeated 'Factors increasing/decreasing' section."""
    markers = ['Factors increasing fraud risk:', 'Factors decreasing fraud risk:']
    positions = []
    for marker in markers:
        start = 0
        while True:
            idx = generated_text.find(marker, start)
            if idx == -1:
                break
            positions.append(idx)
            start = idx + 1
    positions = sorted(positions)
    
    if len(positions) > 2:
        cutoff = positions[2]
        return generated_text[:cutoff].strip()
    return generated_text.strip()


primary = get_primary_response(generated)
print(primary)

Classification: Legitimate
Factors decreasing fraud risk: an internal card identifier (card1) with value 14502; billing region code 143.0; an internal card identifier (card2) with value 191.0; an engineered risk indicator (D2) with value 505.0; an engineered risk indicator (C13) with value 16.0.


In [13]:
def compute_faithfullness_overlap(sample, all_feature_names, k=5):
    full_text = sample['text']
    prompt_section = full_text.split("[/INST]")[0]
    transaction_part = "Transaction details:" + prompt_section.split("Transaction details:")[-1]

    given_features = extract_mentioned_features(transaction_part, all_feature_names)
    generated = generate_response(transaction_part)
    primary_response = get_primary_response(generated)

    mentioned_features = extract_mentioned_features(primary_response, all_feature_names)

    if len(given_features) == 0:
        return None

    correctly_mentioned = set(mentioned_features) & set(given_features)
    fabricated = set(mentioned_features) - set(given_features)

    return{
        'given_count': len(given_features),
        'mentioned_count': len(mentioned_features),
        'correct_overlap': len(correctly_mentioned),
        'fabricated_count': len(fabricated),
        'recall': len(correctly_mentioned) / len(given_features) if given_features else None,
        'fabricated_features': list(fabricated),
    }

result = compute_faithfullness_overlap(val_formatted[0], all_cols)
print(result)


{'given_count': 5, 'mentioned_count': 4, 'correct_overlap': 2, 'fabricated_count': 2, 'recall': 0.4, 'fabricated_features': ['M5', 'card1']}


In [14]:
import time

results = []
start = time.time()
for i, sample in enumerate(val_formatted[:50]):
    r = compute_faithfullness_overlap(sample, all_cols)
    if r is not None:
        r['index'] = i
        results.append(r)
    if (i+1) % 10 == 0:
        print(f"Processed {i+1}/50. Time: {time.time()-start:.1f}s")

import pandas as pd
results_df = pd.DataFrame(results)
print(f"\nMean recall: {results_df['recall'].mean():.3f}")
print(f"% with zero fabrication: {(results_df['fabricated_count']==0).mean()*100:.1f}%")
print(f"Mean fabricated features per example: {results_df['fabricated_count'].mean():.2f}")

Processed 10/50. Time: 107.4s
Processed 20/50. Time: 213.4s
Processed 30/50. Time: 316.9s
Processed 40/50. Time: 430.6s
Processed 50/50. Time: 533.4s

Mean recall: 0.620
% with zero fabrication: 80.0%
Mean fabricated features per example: 0.22


In [15]:
low_recall = results_df[results_df['recall'] < 0.6].sort_values('recall')
print(f"Examples with recall < 0.6: {len(low_recall)} out of {len(results_df)}")
print(low_recall[['index', 'given_count', 'mentioned_count', 'recall']])

Examples with recall < 0.6: 12 out of 50
    index  given_count  mentioned_count  recall
35     35            5                0     0.0
34     34            5                1     0.2
37     37            5                1     0.2
0       0            5                4     0.4
9       9            5                2     0.4
19     19            5                2     0.4
7       7            5                3     0.4
5       5            5                2     0.4
30     30            5                3     0.4
25     25            5                3     0.4
45     45            5                2     0.4
49     49            5                2     0.4


In [16]:

results_large = []
start = time.time()
for i, sample in enumerate(val_formatted[:200]):
    r = compute_faithfullness_overlap(sample, all_cols)
    if r is not None:
        r['index'] = i
        results_large.append(r)
    if (i+1) % 25 == 0:
        print(f"Processed {i+1}/200. Time: {time.time()-start:.1f}s")

results_large_df = pd.DataFrame(results_large)
print(f"\nMean recall: {results_large_df['recall'].mean():.3f}")
print(f"% with zero fabrication: {(results_large_df['fabricated_count']==0).mean()*100:.1f}%")
print(f"Mean fabricated features per example: {results_large_df['fabricated_count'].mean():.2f}")

results_large_df.to_csv('outputs/faithfulness_eval_200.csv', index=False)
print("Saved.")

d:\backup of old laptop\F Folder\Sir Tauseef\airllm_env\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Processed 25/200. Time: 275.4s
Processed 50/200. Time: 540.4s
Processed 75/200. Time: 804.2s
Processed 100/200. Time: 1070.2s
Processed 125/200. Time: 1337.0s
Processed 150/200. Time: 1598.9s
Processed 175/200. Time: 1861.4s
Processed 200/200. Time: 2131.5s

Mean recall: 0.630
% with zero fabrication: 83.5%
Mean fabricated features per example: 0.18
Saved.
